# Single-chain context

In [39]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from types import SimpleNamespace

from cli import Cli
from dataset_generator.types import DATASET_TYPE

# Define the absolute path for the root directory for outputs here (and if possible, make sure to have a prepared _dataset folder)
OUTPUTS_ROOT = Path("/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge")

## Test runs

The next cells will run the model training for different pooling layer strategies.

This process can take a long time to finish.

**If you already have the models trained, you can skip this section and go to the evaluation section.**

In [ ]:
dataset_path = OUTPUTS_ROOT.absolute() / "_dataset"
fixed_args = {
    "device": "cpu",  # Change to "cuda" if you have a GPU and want to use it
    "dataset": DATASET_TYPE.SINGLE,
    "load_data": str(dataset_path),
    "new_dataset": None,
    "force_reload": False,
    "num_workers": 0,
    "tags": "deepsad,tuningpooling",

    "num_epochs": 100,
    "learning_rate": 0.005,
    "augment_factor": 0,
    "dme_threshold": 0.5,
    "rm_semantic_fusion": True,
    "early_stopping": 15,
    "kfolds": 5,
    "first_layer_channels": 128,
    "hidden_channels": 64,
    "dropout": 0.5,
    "input_drop": 0.0,
    "att_drop": 0.0,
    "n_fp_layers": 2,
    "n_mlp_layers": 2,
    "activation": "relu",
    "residual": False,
    "random_seed": 42,
    "test_split": 0.15,
    "rep_dim": 64,
    "eta": 1.0,
    "deep_sad_eps": 1.0,
    "grad_clip": 1.0,
    "weight_decay": 1e-6,
    "ae_pretrain": False,
}

In [ ]:
# Run for mean pooling
pooling = "mean"
Cli.train_model(SimpleNamespace(
    **fixed_args,
    pooling=pooling
))

In [ ]:
# Run for max pooling
pooling = "max"
Cli.train_model(SimpleNamespace(
    **fixed_args,
    pooling=pooling
))

In [ ]:
# Run for sum pooling
pooling = "sum"
Cli.train_model(SimpleNamespace(
    **fixed_args,
    pooling=pooling
))

In [ ]:
# Run for mean+max pooling
pooling = "mean_max"
Cli.train_model(SimpleNamespace(
    **fixed_args,
    pooling=pooling
))

## Results processing

In [40]:
# Obtain all relevant run directories that match the pattern and contain a params.yaml file
run_dirs = sorted(
    p for p in OUTPUTS_ROOT.glob("train_epochs*")
    if p.is_dir() and (p / "params.yaml").exists()
)

print(f"Found {len(run_dirs)} run directories")
for p in run_dirs:
    print(p)

Found 4 run directories
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_deepsadbased__202608051947
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_poolmax_nosf_es15_deepsadbased_tuningpooling__202608051954
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_poolmean_max_nosf_es15_deepsadbased_tuningpooling__202608052008
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_poolsum_nosf_es15_de

In [41]:
import yaml
import pandas as pd

def load_run_config(run_dir: Path) -> dict:
    TOTAL_METAPATHS = 6475  # Total number of metapaths in the dataset
    with open(run_dir / "params.yaml") as f:
        doc = yaml.safe_load(f)

    return {
        "run_dir": run_dir,
        "run_name": run_dir.name,
        "pooling": doc["params"]["pooling"],
    }

configs = [load_run_config(p) for p in run_dirs]
configs_df = pd.DataFrame(configs)
configs_df

,run_dir,run_name,pooling
0,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.005_nosf_es15_dee...,mean
1,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.005_poolmax_nosf_...,max
2,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.005_poolmean_max_...,mean_max
3,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.005_poolsum_nosf_...,sum


In [42]:
def load_test_metrics(run_dir: Path) -> dict:
    df = pd.read_csv(run_dir / "test_metrics.csv")
    metric_cols = [c for c in df.columns if c != "fold"]

    result = {"run_dir": run_dir}
    for col in metric_cols:
        result[f"{col}_mean"] = df[col].mean()
        result[f"{col}_std"] = df[col].std(ddof=0)
    
    result["precision_macro_mean"] = ((df["precision_normal"] + df["precision_anomaly"]) / 2).mean()
    result["precision_macro_std"] = ((df["precision_normal"] + df["precision_anomaly"]) / 2).std(ddof=0)
    result["recall_macro_mean"] = ((df["recall_normal"] + df["recall_anomaly"]) / 2).mean()
    result["recall_macro_std"] = ((df["recall_normal"] + df["recall_anomaly"]) / 2).std(ddof=0)
    result["f1_macro_mean"] = ((df["f1_normal"] + df["f1_anomaly"]) / 2).mean()
    result["f1_macro_std"] = ((df["f1_normal"] + df["f1_anomaly"]) / 2).std(ddof=0)
    return result

metrics = [load_test_metrics(p) for p in run_dirs]
metrics_df = pd.DataFrame(metrics)
metrics_df

,run_dir,test_loss_mean,test_loss_std,accuracy_mean,accuracy_std,precision_normal_mean,precision_normal_std,recall_normal_mean,recall_normal_std,f1_normal_mean,...,mcc_mean,mcc_std,threshold_mean,threshold_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std
0,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,5.743141,7.962240,0.951445,0.005895,0.998485,0.003030,0.940580,0.008451,0.968639,...,0.869306,0.013354,0.140854,0.066711,0.904221,0.009835,0.967433,0.005066,0.930554,0.007676
1,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,7.797383,7.127018,0.923699,0.049509,0.985071,0.016326,0.918841,0.065201,0.949325,...,0.806540,0.092046,0.146275,0.111413,0.878761,0.057688,0.930849,0.038230,0.895308,0.056758
2,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,4.196619,4.538308,0.920231,0.028412,0.983831,0.016939,0.915942,0.046422,0.947673,...,0.792700,0.050482,0.265278,0.298210,0.869535,0.040078,0.926542,0.021943,0.888699,0.032294
3,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,2.937151,3.590408,0.898266,0.016185,0.969110,0.011674,0.901449,0.023188,0.933809,...,0.724870,0.037815,0.110162,0.070826,0.834106,0.024362,0.893582,0.019385,0.856702,0.019954


In [43]:
def load_fold_durations(run_dir: Path) -> dict:
    df = pd.read_csv(run_dir / "fold_durations.csv")
    return {
        "run_dir": run_dir,
        "fold_time_mean": df["time"].mean(),
        "fold_time_std": df["time"].std(ddof=0),  # match summary.txt convention
        "fold_time_total": df["time"].sum(),
    }

durations = [load_fold_durations(p) for p in run_dirs]
durations_df = pd.DataFrame(durations)
durations_df


,run_dir,fold_time_mean,fold_time_std,fold_time_total
0,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,77.896,17.181736,389.48
1,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,82.650,32.678488,413.25
2,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,63.146,9.293074,315.73
3,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,70.034,33.622929,350.17


In [44]:
comparison_df = (
    configs_df
    .merge(metrics_df, on="run_dir")
    .merge(durations_df, on="run_dir")
)
comparison_df["pooling"] = pd.Categorical(comparison_df["pooling"], categories=["mean", "max", "sum", "mean_max"])
comparison_df = (
    comparison_df
    .sort_values(["pooling"])
    .reset_index(drop=True)
)
comparison_df

,run_dir,run_name,pooling,test_loss_mean,test_loss_std,accuracy_mean,accuracy_std,precision_normal_mean,precision_normal_std,recall_normal_mean,...,threshold_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std,fold_time_mean,fold_time_std,fold_time_total
0,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.005_nosf_es15_dee...,mean,5.743141,7.962240,0.951445,0.005895,0.998485,0.003030,0.940580,...,0.066711,0.904221,0.009835,0.967433,0.005066,0.930554,0.007676,77.896,17.181736,389.48
1,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.005_poolmax_nosf_...,max,7.797383,7.127018,0.923699,0.049509,0.985071,0.016326,0.918841,...,0.111413,0.878761,0.057688,0.930849,0.038230,0.895308,0.056758,82.650,32.678488,413.25
2,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.005_poolsum_nosf_...,sum,2.937151,3.590408,0.898266,0.016185,0.969110,0.011674,0.901449,...,0.070826,0.834106,0.024362,0.893582,0.019385,0.856702,0.019954,70.034,33.622929,350.17
3,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.005_poolmean_max_...,mean_max,4.196619,4.538308,0.920231,0.028412,0.983831,0.016939,0.915942,...,0.298210,0.869535,0.040078,0.926542,0.021943,0.888699,0.032294,63.146,9.293074,315.73


In [45]:
trimmed_df = comparison_df[[
    "pooling",
    "accuracy_mean",
    "accuracy_std",
    "precision_macro_mean",
    "precision_macro_std",
    "recall_macro_mean",
    "recall_macro_std",
    "f1_macro_mean",
    "f1_macro_std",
]]
trimmed_df

,pooling,accuracy_mean,accuracy_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std
0,mean,0.951445,0.005895,0.904221,0.009835,0.967433,0.005066,0.930554,0.007676
1,max,0.923699,0.049509,0.878761,0.057688,0.930849,0.038230,0.895308,0.056758
2,sum,0.898266,0.016185,0.834106,0.024362,0.893582,0.019385,0.856702,0.019954
3,mean_max,0.920231,0.028412,0.869535,0.040078,0.926542,0.021943,0.888699,0.032294


In [46]:
PCT_SCALE_EXCLUDE = {"fold_time"}  # keep seconds as seconds, not *100

def format_stat(mean, std, base):
    scale = 1 if base in PCT_SCALE_EXCLUDE else 100
    return f"{mean * scale:.2f} \u00b1 {std * scale:05.2f}"

metric_bases = [c[:-5] for c in trimmed_df.columns if c.endswith("_mean")]

display_df2 = pd.DataFrame(index=trimmed_df.index)
for base in metric_bases:
    display_df2[base] = trimmed_df.apply(
        lambda row: format_stat(row[f"{base}_mean"], row[f"{base}_std"], base), axis=1
    )

display_df2.insert(0, "pooling", trimmed_df["pooling"])
display_df2 = display_df2.set_index("pooling")

final_table = display_df2
final_table


,accuracy,precision_macro,recall_macro,f1_macro
pooling,,,,
mean,95.14 ± 00.59,90.42 ± 00.98,96.74 ± 00.51,93.06 ± 00.77
max,92.37 ± 04.95,87.88 ± 05.77,93.08 ± 03.82,89.53 ± 05.68
sum,89.83 ± 01.62,83.41 ± 02.44,89.36 ± 01.94,85.67 ± 02.00
mean_max,92.02 ± 02.84,86.95 ± 04.01,92.65 ± 02.19,88.87 ± 03.23


In [ ]:
# Turn the unicode ± into LaTeX math syntax, and escape underscores in row labels
import re

METRIC_LABELS = {
    "pooling": "Pooling Type",
    "accuracy": "Accuracy (\\%)",
    "precision_macro": "Precision (\\%)",
    "recall_macro": "Recall (\\%)",
    "f1_macro": "F1-Score (\\%)",
}

POOLING_LABELS = {
    "mean": "Mean Pooling",
    "max": "Max Pooling",
    "sum": "Sum Pooling",
    "mean_max": "Mean + Max Pooling",
}

def to_math_pm(s: str) -> str:
    return re.sub(r"(-?[\d.]+) ± (-?[\d.]+)", r"\1 $\\pm$ \2", s)

latex_ready = final_table.map(to_math_pm)
latex_ready = latex_ready.reset_index()  # turn the "pooling" index back into a plain column
latex_ready = latex_ready.rename(columns=METRIC_LABELS)
latex_ready["Pooling Type"] = latex_ready["Pooling Type"].map(POOLING_LABELS)

latex_str = latex_ready.to_latex(
    index=False,  # drop the numeric row index; "Pooling Type" is now a real column
    escape=False,  # cell values already contain LaTeX ($\pm$), don't double-escape
    column_format="l" + "r" * (len(latex_ready.columns) - 1),
    caption="Effect of the DME threshold on test-set performance",
    label="tab:ablation_semantic_fusion",
)
print(latex_str)


\begin{table}
\caption{Effect of the DME threshold on test-set performance}
\label{tab:ablation_semantic_fusion}
\begin{tabular}{lrrrr}
\toprule
Pooling Type & Accuracy (\%) & Precision (\%) & Recall(\%) & F1-Score (\%) \\
\midrule
Mean Pooling & 95.14 $\pm$ 00.59 & 90.42 $\pm$ 00.98 & 96.74 $\pm$ 00.51 & 93.06 $\pm$ 00.77 \\
Max Pooling & 92.37 $\pm$ 04.95 & 87.88 $\pm$ 05.77 & 93.08 $\pm$ 03.82 & 89.53 $\pm$ 05.68 \\
Sum Pooling & 89.83 $\pm$ 01.62 & 83.41 $\pm$ 02.44 & 89.36 $\pm$ 01.94 & 85.67 $\pm$ 02.00 \\
Mean + Max Pooling & 92.02 $\pm$ 02.84 & 86.95 $\pm$ 04.01 & 92.65 $\pm$ 02.19 & 88.87 $\pm$ 03.23 \\
\bottomrule
\end{tabular}
\end{table}



# Cross-chain context

In [49]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from types import SimpleNamespace

from cli import Cli
from dataset_generator.types import DATASET_TYPE

# Define the absolute path for the root directory for outputs here (and if possible, make sure to have a prepared _dataset folder)
OUTPUTS_ROOT = Path("/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge")

## Test runs

The next cells will run the model training for different pooling layer strategies.

This process can take a long time to finish.

**If you already have the models trained, you can skip this section and go to the evaluation section.**

In [29]:
dataset_path = OUTPUTS_ROOT.absolute() / "_dataset"
fixed_args = {
    "device": "cpu",  # Change to "cuda" if you have a GPU and want to use it
    "dataset": DATASET_TYPE.CCTX,
    "load_data": str(dataset_path),
    "new_dataset": None,
    "force_reload": False,
    "num_workers": 0,
    "tags": "deepsad,tuningpooling",

    "num_epochs": 100,
    "learning_rate": 0.002, # Notice that the new model uses a lower learning rate compared to the single-chain context
    "augment_factor": 0,
    "dme_threshold": 0.5,
    "rm_semantic_fusion": True,
    "early_stopping": 15,
    "kfolds": 5,
    "first_layer_channels": 128,
    "hidden_channels": 64,
    "dropout": 0.5,
    "input_drop": 0.0,
    "att_drop": 0.0,
    "n_fp_layers": 2,
    "n_mlp_layers": 2,
    "activation": "relu",
    "residual": False,
    "random_seed": 42,
    "test_split": 0.15,
    "rep_dim": 64,
    "eta": 1.0,
    "deep_sad_eps": 1.0,
    "grad_clip": 1.0,
    "weight_decay": 1e-6,
    "ae_pretrain": False,
}

In [ ]:
# Run for mean pooling
pooling = "mean"
Cli.train_model(SimpleNamespace(
    **fixed_args,
    pooling=pooling
))

In [ ]:
# Run for max pooling
pooling = "max"
Cli.train_model(SimpleNamespace(
    **fixed_args,
    pooling=pooling
))

In [ ]:
# Run for sum pooling
pooling = "sum"
Cli.train_model(SimpleNamespace(
    **fixed_args,
    pooling=pooling
))

In [ ]:
# Run for mean+max pooling
pooling = "mean_max"
Cli.train_model(SimpleNamespace(
    **fixed_args,
    pooling=pooling
))

## Results processing

In [50]:
# Obtain all relevant run directories that match the pattern and contain a params.yaml file
run_dirs = sorted(
    p for p in OUTPUTS_ROOT.glob("train_epochs*")
    if p.is_dir() and (p / "params.yaml").exists()
)

print(f"Found {len(run_dirs)} run directories")
for p in run_dirs:
    print(p)

Found 4 run directories
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.002_nosf_es15_deepsadbased__202608061807
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.002_poolmax_nosf_es15_deepsadbased_tuningpooling__202608062018
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.002_poolmean_max_nosf_es15_deepsadbased_tuningpooling__202608062103
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-internaltx-eval-tuningpooling-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.002_poolsum_nosf_es15_deepsadbas

In [51]:
import yaml
import pandas as pd

def load_run_config(run_dir: Path) -> dict:
    TOTAL_METAPATHS = 6475  # Total number of metapaths in the dataset
    with open(run_dir / "params.yaml") as f:
        doc = yaml.safe_load(f)

    return {
        "run_dir": run_dir,
        "run_name": run_dir.name,
        "pooling": doc["params"]["pooling"],
    }

configs = [load_run_config(p) for p in run_dirs]
configs_df = pd.DataFrame(configs)
configs_df

,run_dir,run_name,pooling
0,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.002_nosf_es15_dee...,mean
1,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.002_poolmax_nosf_...,max
2,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.002_poolmean_max_...,mean_max
3,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.002_poolsum_nosf_...,sum


In [52]:
def load_test_metrics(run_dir: Path) -> dict:
    df = pd.read_csv(run_dir / "test_metrics.csv")
    metric_cols = [c for c in df.columns if c != "fold"]

    result = {"run_dir": run_dir}
    for col in metric_cols:
        result[f"{col}_mean"] = df[col].mean()
        result[f"{col}_std"] = df[col].std(ddof=0)
    
    result["precision_macro_mean"] = ((df["precision_normal"] + df["precision_anomaly"]) / 2).mean()
    result["precision_macro_std"] = ((df["precision_normal"] + df["precision_anomaly"]) / 2).std(ddof=0)
    result["recall_macro_mean"] = ((df["recall_normal"] + df["recall_anomaly"]) / 2).mean()
    result["recall_macro_std"] = ((df["recall_normal"] + df["recall_anomaly"]) / 2).std(ddof=0)
    result["f1_macro_mean"] = ((df["f1_normal"] + df["f1_anomaly"]) / 2).mean()
    result["f1_macro_std"] = ((df["f1_normal"] + df["f1_anomaly"]) / 2).std(ddof=0)
    return result

metrics = [load_test_metrics(p) for p in run_dirs]
metrics_df = pd.DataFrame(metrics)
metrics_df

,run_dir,test_loss_mean,test_loss_std,accuracy_mean,accuracy_std,precision_normal_mean,precision_normal_std,recall_normal_mean,recall_normal_std,f1_normal_mean,...,mcc_mean,mcc_std,threshold_mean,threshold_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std
0,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,1.419993,2.572114,0.978610,0.006764,0.987472,0.006510,0.989655,0.009888,0.988505,...,0.838027,0.044787,15.696655,22.749816,0.931705,0.046426,0.910212,0.041661,0.916359,0.023679
1,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,2.568346,3.994736,0.954011,0.047180,0.990632,0.004365,0.959770,0.052799,0.974135,...,0.768454,0.141444,63.236588,124.859788,0.862705,0.110998,0.918347,0.029119,0.872709,0.089290
2,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,0.116461,0.055463,0.976471,0.018709,0.987343,0.009244,0.987356,0.011144,0.987346,...,0.819959,0.138872,0.451239,0.439691,0.911071,0.072596,0.909063,0.066875,0.909856,0.069546
3,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,0.105163,0.069219,0.978610,0.008284,0.989680,0.004188,0.987356,0.011144,0.988465,...,0.843383,0.043250,1.181734,1.840650,0.921463,0.046604,0.924447,0.025441,0.919874,0.023824


In [53]:
def load_fold_durations(run_dir: Path) -> dict:
    df = pd.read_csv(run_dir / "fold_durations.csv")
    return {
        "run_dir": run_dir,
        "fold_time_mean": df["time"].mean(),
        "fold_time_std": df["time"].std(ddof=0),  # match summary.txt convention
        "fold_time_total": df["time"].sum(),
    }

durations = [load_fold_durations(p) for p in run_dirs]
durations_df = pd.DataFrame(durations)
durations_df


,run_dir,fold_time_mean,fold_time_std,fold_time_total
0,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,185.444,32.417810,927.22
1,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,223.528,106.015011,1117.64
2,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,214.960,39.186509,1074.80
3,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,281.856,58.711656,1409.28


In [54]:
comparison_df = (
    configs_df
    .merge(metrics_df, on="run_dir")
    .merge(durations_df, on="run_dir")
)
comparison_df["pooling"] = pd.Categorical(comparison_df["pooling"], categories=["mean", "max", "sum", "mean_max"])
comparison_df = (
    comparison_df
    .sort_values(["pooling"])
    .reset_index(drop=True)
)
comparison_df

,run_dir,run_name,pooling,test_loss_mean,test_loss_std,accuracy_mean,accuracy_std,precision_normal_mean,precision_normal_std,recall_normal_mean,...,threshold_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std,fold_time_mean,fold_time_std,fold_time_total
0,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.002_nosf_es15_dee...,mean,1.419993,2.572114,0.978610,0.006764,0.987472,0.006510,0.989655,...,22.749816,0.931705,0.046426,0.910212,0.041661,0.916359,0.023679,185.444,32.417810,927.22
1,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.002_poolmax_nosf_...,max,2.568346,3.994736,0.954011,0.047180,0.990632,0.004365,0.959770,...,124.859788,0.862705,0.110998,0.918347,0.029119,0.872709,0.089290,223.528,106.015011,1117.64
2,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.002_poolsum_nosf_...,sum,0.105163,0.069219,0.978610,0.008284,0.989680,0.004188,0.987356,...,1.840650,0.921463,0.046604,0.924447,0.025441,0.919874,0.023824,281.856,58.711656,1409.28
3,/home/gonca/Documents/GitHub/IST-Thesis/Bridge...,train_epochs100_augment0_lr0.002_poolmean_max_...,mean_max,0.116461,0.055463,0.976471,0.018709,0.987343,0.009244,0.987356,...,0.439691,0.911071,0.072596,0.909063,0.066875,0.909856,0.069546,214.960,39.186509,1074.80


In [55]:
trimmed_df = comparison_df[[
    "pooling",
    "accuracy_mean",
    "accuracy_std",
    "precision_macro_mean",
    "precision_macro_std",
    "recall_macro_mean",
    "recall_macro_std",
    "f1_macro_mean",
    "f1_macro_std",
]]
trimmed_df

,pooling,accuracy_mean,accuracy_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std
0,mean,0.978610,0.006764,0.931705,0.046426,0.910212,0.041661,0.916359,0.023679
1,max,0.954011,0.047180,0.862705,0.110998,0.918347,0.029119,0.872709,0.089290
2,sum,0.978610,0.008284,0.921463,0.046604,0.924447,0.025441,0.919874,0.023824
3,mean_max,0.976471,0.018709,0.911071,0.072596,0.909063,0.066875,0.909856,0.069546


In [56]:
PCT_SCALE_EXCLUDE = {"fold_time"}  # keep seconds as seconds, not *100

def format_stat(mean, std, base):
    scale = 1 if base in PCT_SCALE_EXCLUDE else 100
    return f"{mean * scale:.2f} \u00b1 {std * scale:05.2f}"

metric_bases = [c[:-5] for c in trimmed_df.columns if c.endswith("_mean")]

display_df2 = pd.DataFrame(index=trimmed_df.index)
for base in metric_bases:
    display_df2[base] = trimmed_df.apply(
        lambda row: format_stat(row[f"{base}_mean"], row[f"{base}_std"], base), axis=1
    )

display_df2.insert(0, "pooling", trimmed_df["pooling"])
display_df2 = display_df2.set_index("pooling")

final_table2 = display_df2
final_table2


,accuracy,precision_macro,recall_macro,f1_macro
pooling,,,,
mean,97.86 ± 00.68,93.17 ± 04.64,91.02 ± 04.17,91.64 ± 02.37
max,95.40 ± 04.72,86.27 ± 11.10,91.83 ± 02.91,87.27 ± 08.93
sum,97.86 ± 00.83,92.15 ± 04.66,92.44 ± 02.54,91.99 ± 02.38
mean_max,97.65 ± 01.87,91.11 ± 07.26,90.91 ± 06.69,90.99 ± 06.95


In [ ]:
# Turn the unicode ± into LaTeX math syntax, and escape underscores in row labels
import re

METRIC_LABELS = {
    "pooling": "Pooling Type",
    "accuracy": "Accuracy (\\%)",
    "precision_macro": "Precision (\\%)",
    "recall_macro": "Recall (\\%)",
    "f1_macro": "F1-Score (\\%)",
}

POOLING_LABELS = {
    "mean": "Mean Pooling",
    "max": "Max Pooling",
    "sum": "Sum Pooling",
    "mean_max": "Mean + Max Pooling",
}

def to_math_pm(s: str) -> str:
    return re.sub(r"(-?[\d.]+) ± (-?[\d.]+)", r"\1 $\\pm$ \2", s)

latex_ready2 = final_table2.map(to_math_pm)
latex_ready2 = latex_ready2.reset_index()  # turn the "pooling" index back into a plain column
latex_ready2 = latex_ready2.rename(columns=METRIC_LABELS)
latex_ready2["Pooling Type"] = latex_ready2["Pooling Type"].map(POOLING_LABELS)

latex_str = latex_ready2.to_latex(
    index=False,  # drop the numeric row index; "Pooling Type" is now a real column
    escape=False,  # cell values already contain LaTeX ($\pm$), don't double-escape
    column_format="l" + "r" * (len(latex_ready2.columns) - 1),
    caption="Effect of the DME threshold on test-set performance",
    label="tab:ablation_semantic_fusion",
)
print(latex_str)


\begin{table}
\caption{Effect of the DME threshold on test-set performance}
\label{tab:ablation_semantic_fusion}
\begin{tabular}{lrrrr}
\toprule
Pooling Type & Accuracy (\%) & Precision (\%) & Recall (\%) & F1-Score (\%) \\
\midrule
Mean Pooling & 97.86 $\pm$ 00.68 & 93.17 $\pm$ 04.64 & 91.02 $\pm$ 04.17 & 91.64 $\pm$ 02.37 \\
Max Pooling & 95.40 $\pm$ 04.72 & 86.27 $\pm$ 11.10 & 91.83 $\pm$ 02.91 & 87.27 $\pm$ 08.93 \\
Sum Pooling & 97.86 $\pm$ 00.83 & 92.15 $\pm$ 04.66 & 92.44 $\pm$ 02.54 & 91.99 $\pm$ 02.38 \\
Mean + Max Pooling & 97.65 $\pm$ 01.87 & 91.11 $\pm$ 07.26 & 90.91 $\pm$ 06.69 & 90.99 $\pm$ 06.95 \\
\bottomrule
\end{tabular}
\end{table}



# Putting it all together

In [64]:
# Combine the single-chain and cross-chain pooling comparison tables into one,
# tagging each block of rows with the context (model) it came from.

def prep_model_table(df: pd.DataFrame, model_label: str) -> pd.DataFrame:
    prepped = df.map(to_math_pm)
    prepped = prepped.reset_index()  # turn the "pooling" index back into a plain column
    prepped = prepped.rename(columns=METRIC_LABELS)
    prepped["Pooling Type"] = prepped["Pooling Type"].map(POOLING_LABELS)
    prepped.insert(0, "Model", model_label)
    return prepped

combined_df = pd.concat([
    prep_model_table(final_table, "DeepSAD-based (Single-Chain)"),
    prep_model_table(final_table2, "DeepSAD-based (Cross-Chain)"),
], ignore_index=True)
combined_df = combined_df.set_index(["Model", "Pooling Type"])
combined_df


Accuracy (\%)  \
Model                        Pooling Type                            
DeepSAD-based (Single-Chain) Mean Pooling        95.14 $\pm$ 00.59   
                             Max Pooling         92.37 $\pm$ 04.95   
                             Sum Pooling         89.83 $\pm$ 01.62   
                             Mean + Max Pooling  92.02 $\pm$ 02.84   
DeepSAD-based (Cross-Chain)  Mean Pooling        97.86 $\pm$ 00.68   
                             Max Pooling         95.40 $\pm$ 04.72   
                             Sum Pooling         97.86 $\pm$ 00.83   
                             Mean + Max Pooling  97.65 $\pm$ 01.87   

                                                    Precision (\%)  \
Model                        Pooling Type                            
DeepSAD-based (Single-Chain) Mean Pooling        90.42 $\pm$ 00.98   
                             Max Pooling         87.88 $\pm$ 05.77   
                             Sum Pooling         83.41 $\pm$ 02.44   
                             Mean + Max Pooling  86.95 $\pm$ 04.01   
DeepSAD-based (Cross-Chain)  Mean Pooling        93.17 $\pm$ 04.64   
                             Max Pooling         86.27 $\pm$ 11.10   
                             Sum Pooling         92.15 $\pm$ 04.66   
                             Mean + Max Pooling  91.11 $\pm$ 07.26   

                                                        Recall(\%)  \
Model                        Pooling Type                            
DeepSAD-based (Single-Chain) Mean Pooling        96.74 $\pm$ 00.51   
                             Max Pooling         93.08 $\pm$ 03.82   
                             Sum Pooling         89.36 $\pm$ 01.94   
                             Mean + Max Pooling  92.65 $\pm$ 02.19   
DeepSAD-based (Cross-Chain)  Mean Pooling        91.02 $\pm$ 04.17   
                             Max Pooling         91.83 $\pm$ 02.91   
                             Sum Pooling         92.44 $\pm$ 02.54   
                             Mean + Max Pooling  90.91 $\pm$ 06.69   

                                                     F1-Score (\%)  
Model                        Pooling Type                           
DeepSAD-based (Single-Chain) Mean Pooling        93.06 $\pm$ 00.77  
                             Max Pooling         89.53 $\pm$ 05.68  
                             Sum Pooling         85.67 $\pm$ 02.00  
                             Mean + Max Pooling  88.87 $\pm$ 03.23  
DeepSAD-based (Cross-Chain)  Mean Pooling        91.64 $\pm$ 02.37  
                             Max Pooling         87.27 $\pm$ 08.93  
                             Sum Pooling         91.99 $\pm$ 02.38  
                             Mean + Max Pooling  90.99 $\pm$ 06.95

In [72]:
# Build the LaTeX table by hand: a single header row (Model + Pooling Type + metrics),
# vertically-centered \multirow cells for the repeated "Model" values, and \midrule
# (instead of \cline) between the two model blocks.

flat_df = combined_df.reset_index()
metric_cols = [c for c in flat_df.columns if c not in ("Model", "Pooling Type")]

header = " & ".join(["Model", "Pooling Type", *metric_cols]) + r" \\"

body_lines = []
models = flat_df["Model"].unique()
for i, model in enumerate(models):
    group = flat_df[flat_df["Model"] == model]
    for j, (_, row) in enumerate(group.iterrows()):
        model_cell = f"\\multirow{{{len(group)}}}{{*}}{{{model}}}" if j == 0 else ""
        cells = [model_cell, row["Pooling Type"], *[str(v) for v in row[metric_cols]]]
        body_lines.append(" & ".join(cells) + r" \\")
    if i < len(models) - 1:
        body_lines.append(r"\midrule")

column_format = "ll" + "c" * len(metric_cols)
body = "\n".join(body_lines)

latex_str = (
    "\\begin{table}\n"
    "\\centering\n"
    "\\resizebox{\\linewidth}{!}{%\n"
    f"\\begin{{tabular}}{{{column_format}}}\n"
    "\\toprule\n"
    f"{header}\n"
    "\\midrule\n"
    f"{body}\n"
    "\\bottomrule\n"
    "\\end{tabular}}%\n"
    "\\caption{Effect of pooling strategy on test-set performance for both models, in both single-chain and cross-chain contexts}\n"
    "\\label{tab:pooling_comparison_combined}\n"
    "\\end{table}\n"
)
print(latex_str)


\begin{table}
\centering
\resizebox{\linewidth}{!}{%
\begin{tabular}{llcccc}
\toprule
Model & Pooling Type & Accuracy (\%) & Precision (\%) & Recall(\%) & F1-Score (\%) \\
\midrule
\multirow{4}{*}{DeepSAD-based (Single-Chain)} & Mean Pooling & 95.14 $\pm$ 00.59 & 90.42 $\pm$ 00.98 & 96.74 $\pm$ 00.51 & 93.06 $\pm$ 00.77 \\
 & Max Pooling & 92.37 $\pm$ 04.95 & 87.88 $\pm$ 05.77 & 93.08 $\pm$ 03.82 & 89.53 $\pm$ 05.68 \\
 & Sum Pooling & 89.83 $\pm$ 01.62 & 83.41 $\pm$ 02.44 & 89.36 $\pm$ 01.94 & 85.67 $\pm$ 02.00 \\
 & Mean + Max Pooling & 92.02 $\pm$ 02.84 & 86.95 $\pm$ 04.01 & 92.65 $\pm$ 02.19 & 88.87 $\pm$ 03.23 \\
\midrule
\multirow{4}{*}{DeepSAD-based (Cross-Chain)} & Mean Pooling & 97.86 $\pm$ 00.68 & 93.17 $\pm$ 04.64 & 91.02 $\pm$ 04.17 & 91.64 $\pm$ 02.37 \\
 & Max Pooling & 95.40 $\pm$ 04.72 & 86.27 $\pm$ 11.10 & 91.83 $\pm$ 02.91 & 87.27 $\pm$ 08.93 \\
 & Sum Pooling & 97.86 $\pm$ 00.83 & 92.15 $\pm$ 04.66 & 92.44 $\pm$ 02.54 & 91.99 $\pm$ 02.38 \\
 & Mean + Max Pooling & 97